In [ ]:
from ultralytics import YOLO
from glob import glob
import cv2
import torch
import numpy as np
import subprocess
import os
import os
from PIL import Image
import numpy as np
import tifffile

In [4]:
## background normalization

input_dir = "test_data/marco_new"        # TIFF folder
output_root = f"{input_dir}_background_processed"  # Output root folder
os.makedirs(output_root, exist_ok=True)

percentile_high = 99.99        # White reference percentile; adjust overall brightness
gamma = 1.2                 # >1 lowers contrast, <1 increases contrast
contrast_factor = 0.8       # 0-1 scales down overall contrast
background_fraction = 1/16  # Proportion of lower-left region used as background

# -------------------- Scan all TIFF files for global high percentile --------------------
all_values = []
for file in os.listdir(input_dir):
    if file.lower().endswith((".tiff", ".tif")):
        data = tifffile.imread(os.path.join(input_dir, file))
        all_values.append(data.flatten())
all_values = np.concatenate(all_values)
global_high = np.percentile(all_values, percentile_high)
print(f"Global high percentile for normalization: {global_high:.2f}")

# -------------------- Batch process TIFF files --------------------
for file in os.listdir(input_dir):
    if file.lower().endswith((".tiff", ".tif")):
        tiff_path = os.path.join(input_dir, file)
        base_name = os.path.splitext(file)[0]

        data = tifffile.imread(tiff_path)
        if data.ndim == 2:
            data = data[np.newaxis, ...]  # Single-frame TIFF: (H, W) -> (1, H, W)
        elif data.ndim != 3:
            raise ValueError(f"Unsupported TIFF shape for {file}: {data.shape}")

        n_frames, H, W = data.shape
        print(f"Processing {file}: frames={n_frames}, size=({H},{W})")

        if n_frames > 1:
            output_dir = os.path.join(output_root, base_name)
            os.makedirs(output_dir, exist_ok=True)

        for i, frame in enumerate(data):
            # ---------- Background estimation (lower-left 1/16 region) ----------
            h_start = int(H * (1 - background_fraction))
            w_end = int(W * background_fraction)
            background_region = frame[h_start:, :w_end]
            background = np.mean(background_region)  # Use np.median for better robustness

            # ---------- Background subtraction + highlight normalization ----------
            norm = (frame - background) / (global_high - background + 1e-8)
            norm = np.clip(norm, 0, 1)

            # ---------- Gamma adjustment ----------
            norm = norm ** gamma

            # ---------- Contrast factor adjustment ----------
            norm = np.clip(norm * contrast_factor, 0, 1)

            # ---------- Save as 16-bit PNG ----------
            norm = (norm * 65535).astype(np.uint16)
            if n_frames == 1:
                output_path = os.path.join(output_root, f"{base_name}.png")
            else:
                output_path = os.path.join(output_dir, f"frame_{i:03d}.png")
            Image.fromarray(norm).save(output_path)

        print(f"Done: {file}")

Global high percentile for normalization: 207.85
Processing frame_000.tif: frames=1, size=(128,128)
Done: frame_000.tif
Processing frame_041.tif: frames=1, size=(128,128)
Done: frame_041.tif
Processing frame_001.tif: frames=1, size=(128,128)
Done: frame_001.tif
Processing frame_032.tif: frames=1, size=(128,128)
Done: frame_032.tif
Processing frame_038.tif: frames=1, size=(128,128)
Done: frame_038.tif
Processing frame_046.tif: frames=1, size=(128,128)
Done: frame_046.tif
Processing frame_071.tif: frames=1, size=(128,128)
Done: frame_071.tif
Processing frame_066.tif: frames=1, size=(128,128)
Done: frame_066.tif
Processing frame_061.tif: frames=1, size=(128,128)
Done: frame_061.tif
Processing frame_031.tif: frames=1, size=(128,128)
Done: frame_031.tif
Processing frame_022.tif: frames=1, size=(128,128)
Done: frame_022.tif
Processing frame_028.tif: frames=1, size=(128,128)
Done: frame_028.tif
Processing frame_027.tif: frames=1, size=(128,128)
Done: frame_027.tif
Processing frame_037.tif: fr

In [9]:
MODEL_PATH = "runs/segment/train_txt_run/weights/best.pt"
output_base_dir = "results"
#SOURCE_ROOT = "test_data"
source_dir = output_root
PRED_SUFFIX = "_pridect"  # keep requested spelling
IMG_SIZE = 512
CONF = 0.25
FPS = 10

# Set MAX_IMAGES to an integer for quick testing, or None for all images.
MAX_IMAGES = None

# Prefer GPU first. If CUDA OOM occurs, auto fallback to CPU.
PREFER_GPU = True

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")
# if not os.path.isdir(SOURCE_ROOT):
#     raise FileNotFoundError(f"Source root not found: {SOURCE_ROOT}")

def collect_images(source_dir):
    images = []
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff", "*.webp"):
        images.extend(glob(os.path.join(source_dir, "**", ext), recursive=True))
    images = sorted(images)
    if MAX_IMAGES is not None:
        images = images[:MAX_IMAGES]
    return images

def to_uint8_bgr(img_raw):
    if img_raw.dtype == np.uint16:
        # If values are already in 0..255 range, keep them directly.
        if int(img_raw.max()) <= 255:
            img = img_raw.astype(np.uint8)
        else:
            mn = float(img_raw.min())
            mx = float(img_raw.max())
            if mx <= mn:
                img = np.zeros_like(img_raw, dtype=np.uint8)
            else:
                img = ((img_raw.astype(np.float32) - mn) * 255.0 / (mx - mn)).clip(0, 255).astype(np.uint8)
    elif img_raw.dtype != np.uint8:
        img = cv2.normalize(img_raw, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    else:
        img = img_raw

    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.ndim == 3 and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    return img

def draw_mask_contours_only(res, bgr, color=(0, 0, 255), thickness=2):
    plotted = bgr.copy()
    if res.masks is None or res.masks.xyn is None:
        return plotted, False

    h, w = plotted.shape[:2]
    drew_any = False
    for poly in res.masks.xyn:
        if poly is None or len(poly) < 3:
            continue
        pts = np.stack([poly[:, 0] * w, poly[:, 1] * h], axis=1).astype(np.int32)
        cv2.polylines(plotted, [pts], isClosed=True, color=color, thickness=thickness)
        drew_any = True
    return plotted, drew_any

def run_predict_and_save(model, valid_images, source_dir, pred_root, device):
    processed = 0
    fallback_plain = 0
    for img_path in valid_images:
        rel = os.path.relpath(img_path, source_dir)
        rel_no_ext = os.path.splitext(rel)[0]
        out_path = os.path.join(pred_root, rel_no_ext + ".png")
        os.makedirs(os.path.dirname(out_path), exist_ok=True)

        raw = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        if raw is None:
            continue
        bgr = to_uint8_bgr(raw)

        res = model.predict(
            source=bgr,
            imgsz=IMG_SIZE,
            conf=CONF,
            save=False,
            verbose=False,
            device=device,
        )[0]

        # Draw contour only: no boxes, no labels, no filled green mask background.
        plotted, drew_any = draw_mask_contours_only(res, bgr)
        if not drew_any:
            fallback_plain += 1

        ok, buf = cv2.imencode('.png', plotted)
        if not ok:
            continue
        with open(out_path, 'wb') as f:
            f.write(buf.tobytes())
        processed += 1
    return processed, fallback_plain

def build_lossless_mp4_from_pngs(pred_root, out_mp4, fps):
    pngs = sorted(glob(os.path.join(pred_root, "**", "*.png"), recursive=True))
    if not pngs:
        return False, "No PNG frames found"

    list_path = os.path.join(pred_root, "_ffmpeg_frames.txt")
    with open(list_path, "w", encoding="utf-8") as f:
        for p in pngs:
            abs_p = os.path.abspath(p).replace("'", "'\\''")
            f.write(f"file '{abs_p}'\n")

    # Use yuv444p lossless first for wider player compatibility.
    cmd_yuv = [
        "ffmpeg", "-y",
        "-r", str(fps),
        "-f", "concat", "-safe", "0", "-i", list_path,
        "-c:v", "libx264",
        "-qp", "0",
        "-preset", "veryslow",
        "-pix_fmt", "yuv444p",
        out_mp4,
    ]

    # Fallback to RGB lossless only if needed.
    cmd_rgb = [
        "ffmpeg", "-y",
        "-r", str(fps),
        "-f", "concat", "-safe", "0", "-i", list_path,
        "-c:v", "libx264rgb",
        "-crf", "0",
        "-preset", "veryslow",
        "-pix_fmt", "rgb24",
        out_mp4,
    ]

    try:
        r = subprocess.run(cmd_yuv, check=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if r.returncode == 0:
            return True, f"Saved: {out_mp4}"

        r2 = subprocess.run(cmd_rgb, check=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if r2.returncode == 0:
            return True, f"Saved: {out_mp4}"

        return False, (r2.stderr or r.stderr).splitlines()[-1] if (r2.stderr or r.stderr) else "ffmpeg failed"
    except FileNotFoundError:
        return False, "ffmpeg not found in PATH"
    finally:
        if os.path.exists(list_path):
            os.remove(list_path)

model = YOLO(MODEL_PATH)
preferred_device = "cuda:0" if (PREFER_GPU and torch.cuda.is_available()) else "cpu"

# source_dirs = sorted([d for d in glob(os.path.join(SOURCE_ROOT, "*")) if os.path.isdir(d)])
# if not source_dirs:
#     # Fallback: process SOURCE_ROOT directly if it has images
#     source_dirs = [SOURCE_ROOT]

total_processed = 0
total_fallback = 0
lossless_mp4_outputs = []


source_images = collect_images(source_dir)
if len(source_images) == 0:
    print(f"[SKIP] No supported images found under: {source_dir}")

valid_images = []
shapes = []
bad_images = []
for p in source_images:
    img = cv2.imread(p, cv2.IMREAD_UNCHANGED)
    if img is None:
        bad_images.append(p)
        continue
    if img.ndim == 2:
        h, w = img.shape
    else:
        h, w = img.shape[:2]
    shapes.append((h, w))
    valid_images.append(p)

if len(valid_images) == 0:
    print(f"[SKIP] No readable images found in: {source_dir}")

min_h = min(s[0] for s in shapes)
max_h = max(s[0] for s in shapes)
min_w = min(s[1] for s in shapes)
max_w = max(s[1] for s in shapes)

pred_name = os.path.basename(os.path.normpath(source_dir)) + PRED_SUFFIX
pred_root = os.path.join(output_base_dir, pred_name)

print("=" * 72)
print(f"Source: {source_dir}")
print(f"Output: {os.path.abspath(pred_root)}")
print(f"Selected images: {len(source_images)}")
print(f"Readable images: {len(valid_images)}")
print(f"Unreadable images: {len(bad_images)}")
print(f"Image size range (HxW): H[{min_h}, {max_h}], W[{min_w}, {max_w}]")
if bad_images:
    print("First unreadable file:", bad_images[0])

used_device = preferred_device
try:
    processed, fallback_plain = run_predict_and_save(
        model=model,
        valid_images=valid_images,
        source_dir=source_dir,
        pred_root=pred_root,
        device=preferred_device,
    )
except RuntimeError as e:
    msg = str(e).lower()
    if "out of memory" in msg or "cuda" in msg:
        print("GPU failed (likely CUDA memory issue). Falling back to CPU...")
        used_device = "cpu"
        processed, fallback_plain = run_predict_and_save(
            model=model,
            valid_images=valid_images,
            source_dir=source_dir,
            pred_root=pred_root,
            device="cpu",
        )
    else:
        raise

total_processed += processed
total_fallback += fallback_plain

mp4_path = os.path.join(output_base_dir, pred_name + "_lossless.mp4")
ok_mp4, mp4_msg = build_lossless_mp4_from_pngs(pred_root, mp4_path, FPS)
if ok_mp4:
    lossless_mp4_outputs.append(mp4_path)
    print(f"Lossless MP4: {mp4_path}")
else:
    print(f"[WARN] Failed to create lossless MP4 for {pred_root}: {mp4_msg}")

print(f"Device used: {used_device}")
print(f"Processed images: {processed}")
print(f"No-mask/plain outputs: {fallback_plain}")

print("\n" + "=" * 72)
print(f"Total processed images: {total_processed}")
print(f"Total no-mask/plain outputs: {total_fallback}")
print(f"Lossless MP4 files: {len(lossless_mp4_outputs)}")
for v in lossless_mp4_outputs:
    print(" -", v)
print("Done. Outputs are saved under results/xxxx_pridect and lossless mp4 next to them.")

Source: test_data/marco_new_background_processed
Output: /home/chenfeng/Biohackathon2026/yolo_model/results/marco_new_background_processed_pridect
Selected images: 30
Readable images: 30
Unreadable images: 0
Image size range (HxW): H[128, 128], W[128, 128]
Lossless MP4: results/marco_new_background_processed_pridect_lossless.mp4
Device used: cuda:0
Processed images: 30
No-mask/plain outputs: 13

Total processed images: 30
Total no-mask/plain outputs: 13
Lossless MP4 files: 1
 - results/marco_new_background_processed_pridect_lossless.mp4
Done. Outputs are saved under results/xxxx_pridect and lossless mp4 next to them.


In [10]:
mp4_path_initial = os.path.join(source_dir, source_dir.split('/')[-1] + "_lossless.mp4")
ok_mp4, mp4_msg = build_lossless_mp4_from_pngs(source_dir, mp4_path_initial, FPS)
if ok_mp4:
    lossless_mp4_outputs.append(mp4_path_initial)
    print(f"Lossless MP4: {mp4_path_initial}")
else:
    print(f"[WARN] Failed to create lossless MP4 for {source_dir}: {mp4_msg}")


Lossless MP4: test_data/marco_new_background_processed/marco_new_background_processed_lossless.mp4


In [13]:
# Combine two generated videos side by side with color-safe lossless encoding
video_left = mp4_path_initial if 'mp4_path_initial' in globals() else None
video_right = mp4_path if 'mp4_path' in globals() else None

if (video_left is None or video_right is None) and 'lossless_mp4_outputs' in globals() and len(lossless_mp4_outputs) >= 2:
    video_left, video_right = lossless_mp4_outputs[0], lossless_mp4_outputs[1]

if video_left is None or video_right is None:
    raise RuntimeError("Could not find both input videos. Run the previous cells first.")
if not os.path.exists(video_left):
    raise FileNotFoundError(f"Left video not found: {video_left}")
if not os.path.exists(video_right):
    raise FileNotFoundError(f"Right video not found: {video_right}")

# True lossless and color-stable output (FFV1 in MKV).
side_by_side_lossless = os.path.join(output_base_dir, pred_name + "_side_by_side_lossless.mkv")

# Left stream is grayscale-derived and right stream is color; force both to RGB24 before hstack.
filter_graph = "[0:v]setpts=PTS-STARTPTS,format=rgb24[v0];[1:v]setpts=PTS-STARTPTS,format=rgb24[v1];[v0][v1]scale2ref=trunc(iw*oh/ih/2)*2:ih[v0s][v1s];[v0s][v1s]hstack=inputs=2[v]"

cmd_lossless = [
    "ffmpeg", "-y",
    "-i", video_left,
    "-i", video_right,
    "-filter_complex", filter_graph,
    "-map", "[v]",
    "-an",
    "-c:v", "ffv1",
    "-level", "3",
    "-g", "1",
    side_by_side_lossless,
]

result = subprocess.run(cmd_lossless, check=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if result.returncode != 0:
    err_last = result.stderr.splitlines()[-1] if result.stderr else "ffmpeg failed"
    raise RuntimeError(f"Failed to create lossless side-by-side video: {err_last}")

print(f"Lossless side-by-side video saved: {side_by_side_lossless}")

# Optional compatibility copy for players that do not support MKV/FFV1.
side_by_side_mp4 = os.path.join(output_base_dir, pred_name + "_side_by_side_compatible.mp4")
cmd_compatible_mp4 = [
    "ffmpeg", "-y",
    "-i", side_by_side_lossless,
    "-an",
    "-c:v", "libx264",
    "-crf", "18",
    "-preset", "slow",
    "-pix_fmt", "yuv420p",
    side_by_side_mp4,
]

result_mp4 = subprocess.run(cmd_compatible_mp4, check=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if result_mp4.returncode == 0:
    print(f"Compatible MP4 saved: {side_by_side_mp4}")
else:
    warn_last = result_mp4.stderr.splitlines()[-1] if result_mp4.stderr else "ffmpeg failed"
    print(f"[WARN] Could not create compatible MP4 copy: {warn_last}")

Lossless side-by-side video saved: results/marco_new_background_processed_pridect_side_by_side_lossless.mkv
Compatible MP4 saved: results/marco_new_background_processed_pridect_side_by_side_compatible.mp4
